# Main analysis
**Political Instability and Narrative Fragmentation as a Coupled System: An AI-Enabled Analysis of Northern Ireland**
Nuno Morgado, Amira Mouakher, Zoltán Oszkár Szántó

This notebook builds the monthly indicators from the GDELT extracts and reproduces the figures and statistics of the main analysis, organised by paper section.

| Notebook part | Paper section | Content |
|---|---|---|
| 0–1 | 3.1 | Data acquisition and indicator construction |
| 2 | 4.1 | Descriptive dynamics and validation |
| 3 | 4.2 | Dynamic regressions and regularized models |
| 4 | 4.3 | Directionality: Granger, VAR/FEVD, Local Projections |
| 5 | 4.4 | Tree-based models and persistence baseline |
| 6 | 4.5 | Latent political–discursive regimes |
| 7 | 4.6 | Embedding-based validation and construct validation |
| 8 | Discussion | Monitoring figure |

All figures are written to `figures/` at 300 dpi. Run the notebook top to bottom with the frozen CSV extracts in `data/` (see the README for the data snapshot note).


In [ ]:
from pathlib import Path
import os
import sys

# Run from the repository root or notebooks/; an explicit override is optional.
_candidate = Path(os.environ.get("NARRATIVE_REPO_DIR", Path.cwd())).resolve()
REPO_DIR = next((p for p in [_candidate, *_candidate.parents]
                 if (p / "requirements.txt").is_file() and (p / "notebooks").is_dir()), None)
if REPO_DIR is None:
    raise FileNotFoundError("Open the notebook inside the repository or set NARRATIVE_REPO_DIR.")
os.chdir(REPO_DIR)
DATA_DIR = REPO_DIR / "data"
FIGURES_DIR = REPO_DIR / "figures"
BASE = str(DATA_DIR)
IN_COLAB = "google.colab" in sys.modules
DATA_DIR.mkdir(exist_ok=True)
FIGURES_DIR.mkdir(exist_ok=True)
print(f"Repository: {REPO_DIR}")

RESULTS_DIR = REPO_DIR / "results"
RESULTS_DIR.mkdir(exist_ok=True)


In [ ]:
# ---------------------------------------------------------------------
# Imports and global configuration
# ---------------------------------------------------------------------

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import statsmodels.api as sm

from scipy import stats
from scipy.stats import entropy, pearsonr, spearmanr
from scipy.spatial.distance import jensenshannon

from statsmodels.tsa.stattools import (
    grangercausalitytests,
    adfuller,
)
from statsmodels.tsa.api import VAR

from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Lasso
from sklearn.ensemble import (
    RandomForestRegressor,
    GradientBoostingRegressor,
)
from sklearn.inspection import (
    permutation_importance,
    PartialDependenceDisplay,
)
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score,
    silhouette_score,
)
from sklearn.cluster import KMeans


SEED = 42


def znorm(series: pd.Series) -> pd.Series:
    """Return the z-score normalisation of a pandas Series."""
    std = series.std()

    if std == 0 or pd.isna(std):
        raise ValueError(
            "Cannot standardise a constant or entirely missing series."
        )

    return (series - series.mean()) / std


def savefig(name: str, extension: str = "png") -> Path:
    """
    Save the current Matplotlib figure in the figures directory.

    Parameters
    ----------
    name
        Output filename without extension.
    extension
        File extension. Defaults to 'png'.

    Returns
    -------
    pathlib.Path
        Path of the saved figure.
    """
    output_path = FIGURES_DIR / f"{name}.{extension}"

    plt.savefig(
        output_path,
        dpi=300,
        bbox_inches="tight",
    )

    print(f"Figure saved to: {output_path}")
    return output_path

## 0. Data acquisition (BigQuery: run once, then reload the CSVs)
Requires a Google Cloud project with BigQuery access to the public GDELT dataset. Frozen extracts are included in `data/`; see the data inventory and checksums.


In [ ]:
EVENTS_QUERY = """
SELECT
  DATE_TRUNC(PARSE_DATE('%Y%m%d', CAST(SQLDATE AS STRING)), MONTH) AS month,
  SUM(CASE WHEN EventRootCode IN ('14','15','18','19','20')
           THEN ABS(GoldsteinScale) ELSE 0 END) AS violence,
  SUM(CASE WHEN EventCode IN ('17','173','174','175')
           THEN ABS(GoldsteinScale) ELSE 0 END) AS repression,
  SUM(CASE WHEN Actor1Type1Code IN ('MIL','POL')
             OR Actor2Type1Code IN ('MIL','POL')
           THEN ABS(GoldsteinScale) ELSE 0 END) AS security,
  COUNT(*) AS total_events,
  SUM(CASE WHEN EventRootCode BETWEEN '01' AND '07'
            AND (Actor1Type1Code IN ('GOV','LEG','POL','MIL')
              OR Actor2Type1Code IN ('GOV','LEG','POL','MIL'))
           THEN 1 ELSE 0 END) AS elite_coop,
  SUM(CASE WHEN EventRootCode BETWEEN '08' AND '13'
            AND (Actor1Type1Code IN ('GOV','LEG','POL','MIL')
              OR Actor2Type1Code IN ('GOV','LEG','POL','MIL'))
           THEN 1 ELSE 0 END) AS elite_conflict
FROM `gdelt-bq.gdeltv2.events`
WHERE ActionGeo_CountryCode = 'UK'
  AND ActionGeo_FullName LIKE '%Northern Ireland%'
GROUP BY month ORDER BY month
"""

KG_QUERY = """
SELECT
  DATE_TRUNC(DATE(PARSE_TIMESTAMP('%Y%m%d%H%M%S',
             CAST(DATE AS STRING))), MONTH) AS month,
  theme,
  COUNT(*) AS mentions
FROM `gdelt-bq.gdeltv2.gkg`,
UNNEST(SPLIT(Themes, ';')) AS theme
WHERE Locations LIKE '%Northern Ireland%'
GROUP BY month, theme ORDER BY month
"""

In [ ]:
RUN_BIGQUERY = False
OVERWRITE_EXISTING = False

GCP_PROJECT_ID = "YOUR_GCP_PROJECT_ID"

EVENTS_FILE = DATA_DIR / "events_monthly_northern_ireland.csv"
KG_FILE = DATA_DIR / "kg_theme_counts_northern_ireland.csv"


In [ ]:
# ---------------------------------------------------------------------
# Optional BigQuery extraction
#
# Existing frozen CSV files are used by default.
# Set RUN_BIGQUERY = True to download fresh data.
# Set OVERWRITE_EXISTING = True to replace existing CSV files.
# ---------------------------------------------------------------------



def check_output_path(
    output_path: Path,
    overwrite: bool = False,
) -> None:
    """Prevent accidental replacement of an existing output file."""
    if output_path.exists() and not overwrite:
        raise FileExistsError(
            f"Output file already exists:\n{output_path}\n\n"
            "Set OVERWRITE_EXISTING = True to replace it."
        )


if RUN_BIGQUERY:

    if GCP_PROJECT_ID == "YOUR_GCP_PROJECT_ID":
        raise ValueError(
            "Replace 'YOUR_GCP_PROJECT_ID' with a valid "
            "Google Cloud project ID."
        )

    check_output_path(
        EVENTS_FILE,
        overwrite=OVERWRITE_EXISTING,
    )
    check_output_path(
        KG_FILE,
        overwrite=OVERWRITE_EXISTING,
    )

    from google.cloud import bigquery

    if IN_COLAB:
        from google.colab import auth
        auth.authenticate_user()

    client = bigquery.Client(
        project=GCP_PROJECT_ID
    )

    print("Running GDELT Events query...")
    events_df = client.query(
        EVENTS_QUERY
    ).to_dataframe()

    print("Running GDELT Knowledge Graph query...")
    kg_df = client.query(
        KG_QUERY
    ).to_dataframe()

    events_df.to_csv(
        EVENTS_FILE,
        index=False,
    )

    kg_df.to_csv(
        KG_FILE,
        index=False,
    )

    print(f"Events data saved to:\n{EVENTS_FILE}")
    print(f"Knowledge Graph data saved to:\n{KG_FILE}")

else:
    print("BigQuery extraction skipped.")
    print("The notebook will use the existing frozen CSV files.")

In [ ]:
# ---------------------------------------------------------------------
# Load frozen CSV extracts
# ---------------------------------------------------------------------

required_files = [
    EVENTS_FILE,
    KG_FILE,
]

missing_files = [
    path for path in required_files
    if not path.exists()
]

if missing_files:
    missing_list = "\n".join(
        f"  - {path}" for path in missing_files
    )

    raise FileNotFoundError(
        "The following required data files are missing:\n"
        f"{missing_list}\n\n"
        "Either add them to the data directory or set "
        "RUN_BIGQUERY = True."
    )

events_monthly = pd.read_csv(
    EVENTS_FILE,
    parse_dates=["month"],
)

kg_theme_counts = pd.read_csv(
    KG_FILE,
    parse_dates=["month"],
)

print(
    f"Events data: {events_monthly.shape[0]} rows, "
    f"{events_monthly.shape[1]} columns"
)

print(
    f"KG theme data: {kg_theme_counts.shape[0]} rows, "
    f"{kg_theme_counts.shape[1]} columns"
)

## 1. Indicator construction (Section 3.1 / Algorithm 1)

In [ ]:
events = pd.read_csv(f"{BASE}/events_monthly_northern_ireland.csv",
                     parse_dates=["month"]).sort_values("month")
kg = pd.read_csv(f"{BASE}/kg_theme_counts_northern_ireland.csv",
                 parse_dates=["month"]).dropna(subset=["theme"])

# Restrict to the study period BEFORE computing z-scores            # <-- AJOUT
events = events[(events["month"] >= "2015-02-01")
                & (events["month"] <= "2025-12-01")].reset_index(drop=True)  # <-- AJOUT

# Political instability — UNIFORM weights (round-1 revision)
events["instability"] = (
    (events["violence"] + events["repression"] + events["security"])
    / events["total_events"].replace(0, np.nan)
).fillna(0)

# Geopolitical agent polarization (share of divergent verbal interactions)
events["elite_total"] = events["elite_coop"] + events["elite_conflict"]
events["geo_ag_polarization"] = np.where(
    events["elite_total"] > 0,
    events["elite_conflict"] / events["elite_total"], np.nan)

# Composite risk index (within-case z-scores, study period only)
events["risk_index"] = (znorm(events["instability"])
                        + znorm(events["geo_ag_polarization"]))

# Narrative volatility — Shannon entropy (base 2) of monthly theme distribution
nv = (kg.groupby("month")
        .apply(lambda x: entropy(x["mentions"], base=2))
        .rename("nv").reset_index())
df = events.merge(nv, on="month", how="inner").sort_values("month")
for c in ["nv", "risk_index", "instability", "geo_ag_polarization"]:
    df[f"{c}_lag1"] = df[c].shift(1)

print(f"{len(df)} months, {df['month'].min():%Y-%m} → {df['month'].max():%Y-%m}")

## Section 4.1 — Descriptive dynamics and validation

In [ ]:
# Figure 1 — instability & narrative fragmentation over time
fig, ax1 = plt.subplots(figsize=(12, 4))
ax1.plot(df["month"], df["instability"], color="steelblue",
         linewidth=1.5, label="Political Instability")
ax1.set_ylabel("Instability"); ax1.set_xlabel("Month")
ax2 = ax1.twinx()
ax2.plot(df["month"], df["nv"], linestyle="--", color="coral",
         linewidth=1.5, label="Narrative Fragmentation")
ax2.set_ylabel("Narrative Fragmentation (Entropy)")
ax1.xaxis.set_major_locator(mdates.YearLocator(1))
ax1.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))
plt.setp(ax1.xaxis.get_majorticklabels(), rotation=45)
l1, lab1 = ax1.get_legend_handles_labels()
l2, lab2 = ax2.get_legend_handles_labels()
ax1.legend(l1 + l2, lab1 + lab2, loc="upper left", fontsize=9)
plt.title("Political Instability and Narrative Fragmentation Over Time")
plt.tight_layout(); savefig("fig01_timeseries"); plt.show()

In [ ]:
# Figure 2 — instability vs NV scatter with linear fit
plt.figure(figsize=(5, 4))
plt.scatter(df["instability"], df["nv"], alpha=0.6)
zfit = np.poly1d(np.polyfit(df["instability"], df["nv"], 1))
xs = np.sort(df["instability"].values)
plt.plot(xs, zfit(xs))
plt.xlabel("Political instability"); plt.ylabel("Narrative fragmentation (Entropy)")
#plt.title("Instability and narrative fragmentation")
plt.tight_layout(); savefig("fig02_scatter_fit"); plt.show()

In [ ]:
# Figure 3 — NV vs event volume
plt.figure(figsize=(5, 4))
plt.scatter(df["total_events"], df["nv"], alpha=0.6)
plt.xlabel("Total number of events"); plt.ylabel("Narrative fragmentation (Entropy)")
#plt.title("Narrative Volatility vs. Event Volume")
plt.tight_layout(); savefig("fig03_nv_vs_volume"); plt.show()

In [ ]:
# Figure 4 — NV distribution
plt.figure(figsize=(5, 4))
plt.hist(df["nv"], bins=20)
plt.xlabel("Narrative fragmentation (Entropy)"); plt.ylabel("Frequency")
#plt.title("Distribution of narrative fragmentation")
plt.tight_layout(); savefig("fig04_nv_hist"); plt.show()

In [ ]:
# Figure 5 — correlation matrix
variables = [
    "instability",
    "geo_ag_polarization",
    "nv",
    "total_events"
]

labels = [
    "Instability",
    "Polarization",
    "Fragmentation",
    "Event volume"
]

corr = df[variables].dropna().corr()

plt.figure(figsize=(6, 5))
plt.imshow(corr, cmap="viridis", vmin=-1, vmax=1)
plt.colorbar(label="Pearson correlation")

plt.xticks(
    range(len(labels)),
    labels,
    rotation=45,
    ha="right"
)
plt.yticks(range(len(labels)), labels)

#plt.title("Correlation Matrix of Political and Narrative Indicators")
plt.tight_layout()
savefig("fig05_corr_matrix")
plt.show()

# Tableau numérique avec les mêmes labels lisibles
corr_readable = corr.copy()
corr_readable.index = labels
corr_readable.columns = labels
print(corr_readable.round(3))

## Section 4.2 — Dynamic regression and regularized models

In [ ]:
def coefplot(res, title, fname):
    """Plot regression coefficients with 95% confidence intervals."""

    rename = {
        "const": "Intercept",
        "risk_index_lag1": "Composite risk index (t−1)",
        "nv_lag1": "Narrative fragmentation (t−1)",
    }

    # Extract coefficients and confidence intervals first
    coef = res.params.copy()
    conf = res.conf_int().copy()
    conf.columns = ["lower", "upper"]

    # Apply readable labels consistently
    labels = [rename.get(name, name) for name in coef.index]
    coef.index = labels
    conf.index = labels

    # Error-bar distances must be positive
    lower_error = coef - conf["lower"]
    upper_error = conf["upper"] - coef

    plt.figure(figsize=(7, 4))

    plt.errorbar(
        coef.index,
        coef.values,
        yerr=[lower_error.values, upper_error.values],
        fmt="o",
        capsize=5,
    )

    plt.axhline(
        0,
        linestyle="--",
        linewidth=1,
    )

    plt.title(title)
    plt.ylabel("Estimated coefficient")
    plt.xlabel("")
    plt.xticks(rotation=15, ha="right")

    plt.tight_layout()
    savefig(fname)
    plt.show()


# ---------------------------------------------------------------------
# Figure 6 — Baseline dynamic regression
# nv_t ~ risk index_(t−1) + nv_(t−1)
# ---------------------------------------------------------------------

model_columns = [
    "nv",
    "nv_lag1",
    "risk_index_lag1",
]

dynA = (
    df[model_columns]
    .apply(pd.to_numeric, errors="coerce")
    .dropna()
)

X_A = sm.add_constant(
    dynA[["risk_index_lag1", "nv_lag1"]],
    has_constant="add",
)

y_A = dynA["nv"]

ols_A = sm.OLS(y_A, X_A).fit()

print(ols_A.summary())

coefplot(
    ols_A,
   title="Baseline dynamic regression (95% CI)",
    fname="fig06_baseline",
)

In [ ]:
VARIABLE_LABELS = {
    "const": "Intercept",
    "risk_index_lag1": "Composite risk index (t−1)",
    "nv_lag1": "Narrative fragmentation (t−1)",
    "instability_lag1": "Political instability (t−1)",
    "geo_ag_polarization_lag1": "Polarization among geopolitical agents (t−1)",
}

In [ ]:
# ---------------------------------------------------------------------
# Figures 7 and 9 — Structural dynamic model
# Restricted to months with observed elite discourse
# ---------------------------------------------------------------------

VARIABLE_LABELS = {
    "const": "Intercept",
    "risk_index_lag1": "Composite risk index (t−1)",
    "nv_lag1": "Narrative fragmentation (t−1)",
    "instability_lag1": "Political instability (t−1)",
    "geo_ag_polarization_lag1": (
        "Polarization among geopolitical agents (t−1)"
    ),
}


def coefplot(res, title, fname):
    """Plot regression coefficients with 95% confidence intervals."""

    coefficients = res.params.copy()
    confidence_intervals = res.conf_int().copy()
    confidence_intervals.columns = ["lower", "upper"]

    # Replace variable names with publication-ready labels
    labels = [
        VARIABLE_LABELS.get(variable, variable)
        for variable in coefficients.index
    ]

    coefficients.index = labels
    confidence_intervals.index = labels

    lower_errors = coefficients - confidence_intervals["lower"]
    upper_errors = confidence_intervals["upper"] - coefficients

    plt.figure(figsize=(9, 4.5))

    plt.errorbar(
        coefficients.index,
        coefficients.values,
        yerr=[
            lower_errors.to_numpy(),
            upper_errors.to_numpy(),
        ],
        fmt="o",
        capsize=5,
    )

    plt.axhline(
        y=0,
        linestyle="--",
        linewidth=1,
    )

    plt.title(title)
    plt.ylabel("Estimated coefficient")
    plt.xlabel("")
    plt.xticks(rotation=15, ha="right")

    plt.tight_layout()
    savefig(fname)
    plt.show()


# Select months with observed elite discourse
structural_columns = [
    "month",
    "nv",
    "nv_lag1",
    "instability_lag1",
    "geo_ag_polarization_lag1",
]

dynB = df.loc[
    df["elite_total"] > 0,
    structural_columns,
].copy()

# Convert model variables to numeric values
numeric_columns = [
    "nv",
    "nv_lag1",
    "instability_lag1",
    "geo_ag_polarization_lag1",
]

dynB[numeric_columns] = dynB[numeric_columns].apply(
    pd.to_numeric,
    errors="coerce",
)

dynB["month"] = pd.to_datetime(
    dynB["month"],
    errors="coerce",
)

dynB = (
    dynB
    .dropna(subset=["month"] + numeric_columns)
    .sort_values("month")
)


# ---------------------------------------------------------------------
# Structural dynamic regression
# ---------------------------------------------------------------------

predictors_B = [
    "instability_lag1",
    "geo_ag_polarization_lag1",
    "nv_lag1",
]

X_B = sm.add_constant(
    dynB[predictors_B],
    has_constant="add",
)

y_B = dynB["nv"]

ols_B = sm.OLS(y_B, X_B).fit()

print(ols_B.summary())


# ---------------------------------------------------------------------
# Figure 7 — Coefficients and 95% confidence intervals
# ---------------------------------------------------------------------

coefplot(
    ols_B,
    title="Structural dynamic regression (95% CI)",
    fname="Figure 7",
)


# ---------------------------------------------------------------------
# Figure 9 — Observed versus predicted narrative fragmentation
# ---------------------------------------------------------------------

dynB["nv_pred"] = ols_B.predict(X_B)

plt.figure(figsize=(10, 4))

plt.plot(
    dynB["month"],
    dynB["nv"],
    label="Observed narrative fragmentation",
)

plt.plot(
    dynB["month"],
    dynB["nv_pred"],
    linestyle="--",
    label="Predicted narrative fragmentation",
)

plt.xlabel("Month")
plt.ylabel("Narrative fragmentation (entropy)")
plt.legend()
plt.tight_layout()

savefig("Figure 9")
plt.show()

In [ ]:
# Figure 8 — LASSO coefficient paths

features = [
    "instability_lag1",
    "geo_ag_polarization_lag1",
    "risk_index_lag1",
    "nv_lag1",
    "violence",
    "repression",
    "security",
    "total_events"
]

labels = [
    "Instability (t−1)",
    "Polarization (t−1)",
    "Risk index (t−1)",
    "Fragmentation (t−1)",
    "Violence",
    "Repression",
    "Security",
    "Event volume"
]

# Préparation des données
lf = (
    df[features + ["nv"]]
    .apply(pd.to_numeric, errors="coerce")
    .dropna()
)

# Standardisation des variables explicatives
Xs = StandardScaler().fit_transform(lf[features])

# Calcul des trajectoires des coefficients
alphas = np.logspace(-3, 1, 120)

coefs = np.array([
    Lasso(alpha=alpha, max_iter=20000)
    .fit(Xs, lf["nv"])
    .coef_
    for alpha in alphas
])

# Figure
plt.figure(figsize=(10, 5.5))

for i, label in enumerate(labels):
    plt.plot(
        alphas,
        coefs[:, i],
        linewidth=1.7,
        label=label
    )

plt.axhline(0, color="black", linewidth=0.8, alpha=0.6)
plt.xscale("log")

plt.xlabel("Regularization strength (alpha)")
plt.ylabel("Coefficient")
#plt.title("LASSO Coefficient Paths")

plt.legend(
    bbox_to_anchor=(1.02, 1),
    loc="upper left",
    frameon=False
)

plt.tight_layout()
savefig("Figure 8")
plt.show()

## Section 4.3 — Directionality and causal assessment

In [ ]:
# Granger causality on first differences (with ADF checks)
d43 = df[["month", "nv", "instability"]].dropna().sort_values("month")
for col, name in [("nv", "NV"), ("instability", "Instability")]:
    adf = adfuller(d43[col])
    print(f"ADF {name}: stat={adf[0]:.3f} p={adf[1]:.4f}")

g = d43.assign(nv_d=d43["nv"].diff(), inst_d=d43["instability"].diff()).dropna()
gc_fwd = grangercausalitytests(g[["nv_d", "inst_d"]], maxlag=3, verbose=False)
gc_rev = grangercausalitytests(g[["inst_d", "nv_d"]], maxlag=3, verbose=False)
for lag in range(1, 4):
    p1 = gc_fwd[lag][0]["ssr_ftest"][1]
    p2 = gc_rev[lag][0]["ssr_ftest"][1]
    print(f"lag {lag}: instability→NV p={p1:.4f} | NV→instability p={p2:.4f}")

In [ ]:

var_data = d43.set_index("month")[["nv", "instability"]].astype(float)
var_model = VAR(var_data)
order = var_model.select_order(maxlags=8)
p_aic = order.aic
print(f"AIC-selected lag order: {p_aic}")
var_fitted = var_model.fit(p_aic)

fwd = var_fitted.test_causality("nv", ["instability"], kind="f")
rev = var_fitted.test_causality("instability", ["nv"], kind="f")
print(f"VAR F-test instability→NV: F={fwd.test_statistic:.2f} p={fwd.pvalue:.3f}")
print(f"VAR F-test NV→instability: F={rev.test_statistic:.2f} p={rev.pvalue:.3f}")

fevd = var_fitted.fevd(periods=12)
dec = fevd.decomp   # (neqs, periods, neqs); variable order = [nv, instability]
for h in [1, 3, 6, 12]:
    print(f"FEVD h={h:2d}: NV variance from instability = {dec[0, h-1, 1]:.1%}"
          f" | instability variance from NV = {dec[1, h-1, 0]:.1%}")

from statsmodels.stats.stattools import durbin_watson
dw = durbin_watson(var_fitted.resid)
pt = var_fitted.test_whiteness(nlags=12)
print(f"Durbin-Watson: nv={dw[0]:.2f}, instability={dw[1]:.2f}; "
      f"Portmanteau p={pt.pvalue:.3f}")


# Forced lag order 4 (the order reported in the submitted manuscript)
var_fitted4 = var_model.fit(4)
fwd4 = var_fitted4.test_causality("nv", ["instability"], kind="f")
rev4 = var_fitted4.test_causality("instability", ["nv"], kind="f")
print(f"VAR(4) instability→NV: F={fwd4.test_statistic:.2f} p={fwd4.pvalue:.3f}")
print(f"VAR(4) NV→instability: F={rev4.test_statistic:.2f} p={rev4.pvalue:.3f}")

In [ ]:
# Robustness: VAR on first-differenced series (NV fails to reject a unit root)
var_d = var_data.diff().dropna()
var_model_d = VAR(var_d)
p_aic_d = var_model_d.select_order(maxlags=8).aic
var_fd = var_model_d.fit(p_aic_d)
f1 = var_fd.test_causality("nv", ["instability"], kind="f")
r1 = var_fd.test_causality("instability", ["nv"], kind="f")
print(f"Differenced VAR (AIC order {p_aic_d}): "
      f"inst→NV F={f1.test_statistic:.2f} p={f1.pvalue:.3f} | "
      f"NV→inst F={r1.test_statistic:.2f} p={r1.pvalue:.3f}")

In [ ]:
dec_d = var_fd.fevd(periods=12).decomp
print(f"Differenced FEVD h=12: NV var. from inst = {dec_d[0, 11, 1]:.1%} | "
      f"inst var. from NV = {dec_d[1, 11, 0]:.1%}")

In [ ]:
# Figure 10 — bidirectional Local Projections (Jordà 2005), HAC errors
n_lags, max_horizon = 3, 12
lp = d43.copy()
lp["nv_z"], lp["inst_z"] = znorm(lp["nv"]), znorm(lp["instability"])
for lag in range(1, n_lags + 1):
    lp[f"ctrl_nv{lag}"] = lp["nv_z"].shift(lag)
    lp[f"ctrl_inst{lag}"] = lp["inst_z"].shift(lag)

def run_lp(outcome, predictor, controls, h):
    tmp = lp.copy()
    tmp["y"] = tmp[outcome].shift(-h)
    tmp = tmp[[predictor] + controls + ["y"]].dropna()
    X = tmp[[predictor] + controls].astype(float)
    X.insert(0, "const", 1.0)
    res = sm.OLS(tmp["y"].astype(float), X).fit(
        cov_type="HAC", cov_kwds={"maxlags": max(h + 1, 1)})
    return res.params[predictor], res.bse[predictor], res.pvalues[predictor]

ctrl_fwd = ([f"ctrl_nv{l}" for l in range(1, n_lags + 1)]
            + [f"ctrl_inst{l}" for l in range(2, n_lags + 1)])
ctrl_rev = ([f"ctrl_inst{l}" for l in range(1, n_lags + 1)]
            + [f"ctrl_nv{l}" for l in range(2, n_lags + 1)])

lp_results = {}
for key, (outcome, pred, ctrls) in {
    "fwd": ("nv_z", "ctrl_inst1", ctrl_fwd),
    "rev": ("inst_z", "ctrl_nv1", ctrl_rev),
}.items():
    rows = []
    for h in range(max_horizon + 1):
        c, se, p = run_lp(outcome, pred, ctrls, h)
        rows.append({"horizon": h, "coef": c, "se": se, "pval": p,
                     "ci95_lo": c - 1.96 * se, "ci95_hi": c + 1.96 * se,
                     "ci90_lo": c - 1.645 * se, "ci90_hi": c + 1.645 * se})
    lp_results[key] = pd.DataFrame(rows)
    sig5 = lp_results[key].query("pval < 0.05")["horizon"].tolist()
    m16 = lp_results[key].query("1 <= horizon <= 6")["coef"].mean()
    print(f"LP {key}: significant (5%) at h={sig5}; mean coef h1-6 = {m16:.3f}")

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
cfg = [("fwd", "steelblue",
        "Response of NF to instability shock\n(Forward: Instability → NV)",
        "Response of narrative fragmentation (z-score)"),
       ("rev", "coral",
        "Response of instability to NF shock\n(Reverse: NV → Instability)",
        "Response of political instability (z-score)")]
for ax, (key, color, title, ylab) in zip(axes, cfg):
    r = lp_results[key]
    ax.plot(r["horizon"], r["coef"], color=color, linewidth=2.5,
            marker="o", markersize=5, label="Point estimate", zorder=5)
    ax.fill_between(r["horizon"], r["ci95_lo"], r["ci95_hi"],
                    alpha=0.15, color=color, label="95% CI")
    ax.fill_between(r["horizon"], r["ci90_lo"], r["ci90_hi"],
                    alpha=0.25, color=color, label="90% CI")
    ax.axhline(0, color="black", linewidth=1.2, linestyle="--")
    ax.set_xlabel("Months after shock"); ax.set_ylabel(ylab)
    ax.set_title(title, fontweight="bold")
    ax.legend(fontsize=9); ax.grid(True, alpha=0.3)
    ax.set_xticks(range(0, max_horizon + 1, 2))
plt.tight_layout(); savefig("fig10_local_projections"); plt.show()

In [ ]:
print(lp_results["fwd"][["horizon", "coef", "pval"]].round(4))
print("min p forward:", round(lp_results["fwd"]["pval"].min(), 4))
print("min p reverse:", round(lp_results["rev"]["pval"].min(), 4))

In [ ]:
# Multiple-testing corrections for the Local Projections:
# Bonferroni across the 13 horizons and a joint Wald test over h = 1 to 6.
from scipy import stats as st

print(f"Bonferroni threshold (0.05 / 13) = {0.05/13:.4f}")
for key in ["fwd", "rev"]:
    print(f"LP {key}: min p across horizons = {lp_results[key]['pval'].min():.4f}")

def joint_wald(y_col, shock, ctrl, h_set=(1, 2, 3, 4, 5, 6)):
    Y = pd.concat({h: lp[y_col].shift(-h) for h in h_set}, axis=1)
    d = pd.concat([lp[[shock] + ctrl], Y], axis=1).dropna()
    X = sm.add_constant(d[[shock] + ctrl].astype(float)).values
    Yv = d[list(h_set)].astype(float).values
    T, k = X.shape; m = Yv.shape[1]
    XtXi = np.linalg.inv(X.T @ X)
    B = XtXi @ X.T @ Yv
    E = Yv - X @ B
    Sig = E.T @ E / (T - k)
    b = B[1, :]                      # the shock is the first regressor after the constant
    Vb = Sig * XtXi[1, 1]
    F = (b @ np.linalg.inv(Vb) @ b) / m
    p = 1 - st.f.cdf(F, m, T - k)
    print(f"Joint Wald ({y_col} <- {shock}, h=1-6): F = {F:.2f}, p = {p:.3f}  (n = {T})")

joint_wald("nv_z", "ctrl_inst1",
           ["ctrl_nv1", "ctrl_nv2", "ctrl_nv3", "ctrl_inst2", "ctrl_inst3"])
joint_wald("inst_z", "ctrl_nv1",
           ["ctrl_inst1", "ctrl_inst2", "ctrl_inst3", "ctrl_nv2", "ctrl_nv3"])


## Section 4.4 — Nonlinear effects: tree-based models

In [ ]:
feature_labels = {
    "nv_lag1": "Fragmentation (t−1)",
    "instability_lag1": "Instability (t−1)",
    "geo_ag_polarization_lag1": "Polarization (t−1)",
    "violence": "Violence",
    "repression": "Repression",
    "security": "Security",
    "total_events": "Event volume"
}

In [ ]:
# Tree-based models: Random Forest and Gradient Boosting, temporal 80/20 holdout
tree_df = df[df["elite_total"] > 0].dropna(
    subset=["nv", "nv_lag1", "instability_lag1",
            "geo_ag_polarization_lag1"]).copy()
feature_cols = ["nv_lag1", "instability_lag1", "geo_ag_polarization_lag1",
                "violence", "repression", "security", "total_events"]
X = tree_df[feature_cols].apply(pd.to_numeric, errors="coerce")
y = pd.to_numeric(tree_df["nv"], errors="coerce")
mask = X.notna().all(axis=1) & y.notna()
X, y, tree_df = X[mask], y[mask], tree_df.loc[mask]

split = int(len(tree_df) * 0.8)
X_tr, X_te = X.iloc[:split], X.iloc[split:]
y_tr, y_te = y.iloc[:split], y.iloc[split:]
months_te = tree_df["month"].iloc[split:]

rf = RandomForestRegressor(n_estimators=500, min_samples_leaf=2,
                           random_state=SEED).fit(X_tr, y_tr)
gb = GradientBoostingRegressor(random_state=SEED).fit(X_tr, y_tr)
pred_rf, pred_gb = rf.predict(X_te), gb.predict(X_te)
for name, pred in [("Random Forest", pred_rf), ("Gradient Boosting", pred_gb)]:
    print(f"{name}: MAE={mean_absolute_error(y_te, pred):.4f}  "
          f"RMSE={np.sqrt(mean_squared_error(y_te, pred)):.4f}  "
          f"R²={r2_score(y_te, pred):.3f}")

best_name, best_model, best_pred = "Random Forest", rf, pred_rf
if mean_absolute_error(y_te, pred_gb) < mean_absolute_error(y_te, pred_rf):
    best_name, best_model, best_pred = "Gradient Boosting", gb, pred_gb

plt.figure(figsize=(10, 4))
plt.plot(months_te, y_te.values, label="Observed NF")
plt.plot(months_te, best_pred, "--", label=f"Predicted NF ({best_name})")
plt.xlabel("Month"); plt.ylabel("Narrative fragmentation (Entropy)")
#plt.title(f"Observed vs predicted narrative fragmentation (Holdout Test: {best_name})")
plt.legend(); plt.tight_layout(); savefig("fig11_rf_holdout"); plt.show()

# Naive lag-1 persistence baseline
naive_pred = X_te["nv_lag1"].astype(float)
print(f"Persistence baseline: MAE={mean_absolute_error(y_te, naive_pred):.4f}  "
      f"RMSE={np.sqrt(mean_squared_error(y_te, naive_pred)):.4f}  "
      f"R²={r2_score(y_te, naive_pred):.3f}")

In [ ]:
# Persistence baseline (naive forecast: previous month's value)
ml = df[["month", "nv", "nv_lag1"]].dropna().sort_values("month").reset_index(drop=True)
split = int(len(ml) * 0.8)          # 104 training months
test = ml.iloc[split:]              # final 26 months = holdout window
mae_p = mean_absolute_error(test["nv"], test["nv_lag1"])
r2_p  = r2_score(test["nv"], test["nv_lag1"])
print(f"Persistence baseline (n_test={len(test)}): MAE={mae_p:.4f}  R²={r2_p:.3f}")

In [ ]:
# Figure 12 — permutation importance

feature_labels = {
    "nv_lag1": "Fragmentation (t−1)",
    "instability_lag1": "Instability (t−1)",
    "geo_ag_polarization_lag1": "Polarization (t−1)",
    "violence": "Violence",
    "repression": "Repression",
    "security": "Security",
    "total_events": "Event volume"
}

perm = permutation_importance(
    best_model,
    X_te,
    y_te,
    n_repeats=30,
    random_state=SEED
)

imp = pd.Series(
    perm.importances_mean,
    index=feature_cols
).sort_values()

# Remplacer les noms techniques par les labels lisibles
imp.index = [feature_labels[feature] for feature in imp.index]

plt.figure(figsize=(7.5, 4.5))

plt.barh(
    imp.index,
    imp.values,
    color="#4c78a8",
    edgecolor="white"
)

plt.xlabel("Mean decrease in predictive performance")
#plt.title(f"Permutation Feature Importance ({best_name})")

plt.grid(
    axis="x",
    linestyle="--",
    linewidth=0.6,
    alpha=0.4
)

plt.tight_layout()
savefig("fig12_perm_importance")
plt.show()

In [ ]:
# Figure 13 — Partial dependence for selected predictors

selected_features = ["security", "total_events"]

fig, ax = plt.subplots(figsize=(10, 4.5))

display = PartialDependenceDisplay.from_estimator(
    best_model,
    X_tr,
    features=selected_features,
    ax=ax,
    line_kw={
        "color": "#1f4e79",
        "linewidth": 2
    }
)

# Labels lisibles
display.axes_[0, 0].set_xlabel("Security")
display.axes_[0, 1].set_xlabel("Event volume")

display.axes_[0, 0].set_ylabel(
    "Partial dependence\nof narrative fragmentation"
)

fig.suptitle(
    f"Partial Dependence of Narrative Fragmentation ({best_name})",
    fontsize=13,
    fontweight="bold"
)

plt.tight_layout()
savefig("fig13_partial_dependence")
plt.show()

## Section 4.5 — Latent political–discursive regimes

In [ ]:
# Cluster-number diagnostics: inertia and silhouette for k = 2 to 6
pol = "geo_ag_polarization"
feats = ["instability", pol]
rdf = df[feats + ["month", "nv"]].dropna().sort_values("month")
scaler = StandardScaler().fit(rdf[feats])
Xs = scaler.transform(rdf[feats])

for k in range(2, 7):
    km = KMeans(n_clusters=k, random_state=SEED, n_init=20)
    lab = km.fit_predict(Xs)
    print(f"k={k}: inertia={km.inertia_:.1f} "
          f"silhouette={silhouette_score(Xs, lab):.3f}")

In [ ]:
# Final clustering (k=3), relabeling to the paper's regime convention, and distributional tests
kmeans = KMeans(n_clusters=3, random_state=SEED, n_init=20)
rdf["regime"] = kmeans.fit_predict(Xs)

# Relabel: Regime 1 = baseline, Regime 2 = acute stress, Regime 3 = polarization
cent_raw = pd.DataFrame(scaler.inverse_transform(kmeans.cluster_centers_),
                        columns=feats)
acute = cent_raw["instability"].idxmax()
polar = cent_raw.drop(index=acute)["geo_ag_polarization"].idxmax()
base = [i for i in range(3) if i not in (acute, polar)][0]
rdf["regime"] = rdf["regime"].map({base: 0, acute: 1, polar: 2})
kmeans.cluster_centers_ = kmeans.cluster_centers_[[base, acute, polar]]

print("regime sizes:", rdf["regime"].value_counts().sort_index().to_dict())

by_regime = [rdf[rdf["regime"] == i]["nv"].values for i in range(3)]
H, p = stats.kruskal(*by_regime)
print(f"Kruskal-Wallis: H={H:.2f} p={p:.4f}")
for i in range(3):
    for j in range(i + 1, 3):
        U, pu = stats.mannwhitneyu(by_regime[i], by_regime[j],
                                   alternative="two-sided")
        print(f"Mann-Whitney R{i+1} vs R{j+1}: U={U:.0f} p={pu:.4f}")

In [ ]:
# Regime profiles: means and medians of the inputs and of narrative fragmentation
print(rdf.groupby("regime")[["instability", "geo_ag_polarization", "nv"]]
      .agg(["mean", "median"]).round(3))


In [ ]:
# Figure 14 — regimes in instability–polarization space
colors = {0: "steelblue", 1: "coral", 2: "green"}
cent = pd.DataFrame(scaler.inverse_transform(kmeans.cluster_centers_),
                    columns=feats)
fig, ax = plt.subplots(figsize=(7, 6))
for i in range(3):
    m = rdf["regime"] == i
    ax.scatter(rdf.loc[m, "instability"], rdf.loc[m, pol],
               c=colors[i], alpha=0.7, s=50,
               label=f"Regime {i+1} (n={m.sum()})")
    ax.scatter(cent.loc[i, "instability"], cent.loc[i, pol],
               c=colors[i], marker="*", s=300, edgecolors="black",
               linewidth=1, zorder=5)
ax.set_xlabel("Political Instability")
ax.set_ylabel("Geopolitical Agent Polarization")
ax.set_title("Latent Political–Discursive Regimes", fontweight="bold")
ax.legend(fontsize=9); ax.grid(True, alpha=0.3)
plt.tight_layout(); savefig("fig14_regimes_scatter"); plt.show()

In [ ]:
# Robustness of the clustering method: GMM/BIC and agreement with K-means
from sklearn.mixture import GaussianMixture
from sklearn.metrics import adjusted_rand_score
bics = {k: GaussianMixture(k, random_state=SEED, n_init=10).fit(Xs).bic(Xs)
        for k in range(2, 7)}
print("BIC:", {k: round(v, 1) for k, v in bics.items()},
      "| BIC-preferred k =", min(bics, key=bics.get))
gmm3 = GaussianMixture(3, random_state=SEED, n_init=10).fit_predict(Xs)
km3 = KMeans(3, random_state=SEED, n_init=20).fit_predict(Xs)
print(f"ARI (GMM3 vs KMeans3) = {adjusted_rand_score(gmm3, km3):.2f}")

In [ ]:
# Figure 15 — NV by regime, with significance brackets
fig, ax = plt.subplots(figsize=(7, 6))
bp = ax.boxplot(by_regime, patch_artist=True,
                medianprops=dict(color="black", linewidth=2))
for patch, c in zip(bp["boxes"], colors.values()):
    patch.set_facecolor(c); patch.set_alpha(0.7)
ymax = max(dd.max() for dd in by_regime)
# Option B — p exacts (imparable, cohérent avec le texte)
for (a, b, txt, yo) in [(1, 2, "p = 0.002", 0.02), (2, 3, "p = 0.003", 0.02),
                        (1, 3, "p = 0.66", 0.12)]:
    ax.plot([a, a, b, b], [ymax + yo, ymax + yo + 0.03,
                           ymax + yo + 0.03, ymax + yo],
            color="black", linewidth=1)
    ax.text((a + b) / 2, ymax + yo + 0.04, txt, ha="center",
            va="bottom", fontsize=11)
ax.set_xticklabels([f"Regime {i+1}\n(n={len(dd)})"
                    for i, dd in enumerate(by_regime)])
ax.set_xlabel("Regime"); ax.set_ylabel("Narrative fragmentation (Entropy)")
#ax.set_title("Narrative Fragmentation Across Political Regimes",fontweight="bold")
ax.grid(True, alpha=0.3, axis="y")
plt.tight_layout(); savefig("fig15_regimes_boxplot"); plt.show()

In [ ]:
# Figure 16 — temporal evolution of regimes
fig, ax = plt.subplots(figsize=(14, 3))
ax.plot(rdf["month"], rdf["regime"] + 1, drawstyle="steps-post",
        color="steelblue", linewidth=1.5)
ax.set_xlim(pd.Timestamp("2015-01-01"), pd.Timestamp("2026-06-01"))
ax.xaxis.set_major_locator(mdates.YearLocator(1))
ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))
plt.setp(ax.xaxis.get_majorticklabels(), rotation=45)
ax.set_yticks([1, 2, 3])
ax.set_yticklabels(["Regime 1", "Regime 2", "Regime 3"])
ax.set_ylim(0.5, 3.5)
ax.set_xlabel("Month"); ax.set_ylabel("Regime")
ax.set_title("Temporal Evolution of Political–Discursive Regimes",
             fontweight="bold")
ax.grid(True, alpha=0.3, axis="x")
plt.tight_layout(); savefig("fig16_regimes_time"); plt.show()

## Section 4.6 — Semantic robustness (embeddings) — Figure 17
Heavy cell (SentenceTransformer): run once, results saved to CSV.

In [ ]:
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_distances

monthly_text = (kg.groupby("month")["theme"]
                   .apply(lambda x: " ".join(x.astype(str))).reset_index())
st_model = SentenceTransformer("all-MiniLM-L6-v2")

def dispersion(text):
     if pd.isna(text) or not str(text).strip():
         return 0.0
     emb = st_model.encode(str(text).split(" "))
     if len(emb) <= 1:
         return 0.0
     centroid = np.mean(emb, axis=0, keepdims=True)
     return cosine_distances(emb, centroid).mean()

monthly_text["LLM_volatility"] = monthly_text["theme"].map(dispersion)
monthly_text = monthly_text.rename(columns={"theme": "text"})
monthly_text.to_csv(f"{BASE}/monthly_news_text_with_volatility.csv", index=False)

In [ ]:
# Figure 17 — plot from the saved dispersion series
sem = pd.read_csv(f"{BASE}/monthly_news_text_with_volatility.csv",
                  parse_dates=["month"]).sort_values("month")
sem["moving_avg"] = sem["LLM_volatility"].rolling(3).mean()
plt.figure(figsize=(14, 7))
plt.plot(sem["month"], sem["LLM_volatility"], "--o", markersize=4,
         alpha=0.4, color="#3498db", label="Monthly Volatility")
plt.plot(sem["month"], sem["moving_avg"], color="#2c3e50", linewidth=2.5,
         label="3-Month Rolling Average")
plt.title("Evolution of Media Narrative Volatility in Northern Ireland")
plt.xlabel("Year"); plt.ylabel("Semantic Dispersion (LLM Volatility)")
plt.grid(True, linestyle="--", alpha=0.5)
plt.legend(loc="upper left"); plt.xticks(rotation=45)
plt.tight_layout(); savefig("fig17_semantic_dispersion"); plt.show()

## Section 4.6 — Construct validation: static entropy vs temporal measures — Figure 18

In [ ]:
# JSD between consecutive monthly theme distributions
months = sorted(kg["month"].unique())
jsd_rows = []
for i in range(1, len(months)):
    prev = kg[kg["month"] == months[i - 1]].set_index("theme")["mentions"]
    curr = kg[kg["month"] == months[i]].set_index("theme")["mentions"]
    all_t = prev.index.union(curr.index)
    pv = prev.reindex(all_t, fill_value=0).values.astype(float)
    qv = curr.reindex(all_t, fill_value=0).values.astype(float)
    pv, qv = pv / pv.sum(), qv / qv.sum()
    jsd_rows.append({"month": months[i], "jsd": jensenshannon(pv, qv)})
jsd_df = pd.DataFrame(jsd_rows)

val = df[["month", "nv", "instability"]].merge(jsd_df, on="month",
                                               how="inner").dropna()
val = val.sort_values("month").reset_index(drop=True)
val["nv_roll_sd_3"] = val["nv"].rolling(3, min_periods=2).std()
val["nv_roll_sd_6"] = val["nv"].rolling(6, min_periods=3).std()
val["nv_diff"] = val["nv"].diff().abs()

for col, lab in [("nv_roll_sd_3", "Rolling SD 3m"),
                 ("nv_roll_sd_6", "Rolling SD 6m"),
                 ("nv_diff", "First difference"), ("jsd", "JSD")]:
    tmp = val[["nv", col]].dropna()
    r, pr = pearsonr(tmp["nv"], tmp[col])
    print(f"corr(entropy, {lab}): r={r:.3f} p={pr:.4f}")

In [ ]:
# Predictive validity of each measure against instability + Figure 18
measures = {"Static\nEntropy": "nv", "Rolling SD\n(3-month)": "nv_roll_sd_3",
            "Rolling SD\n(6-month)": "nv_roll_sd_6",
            "First\nDifference": "nv_diff", "JSD": "jsd"}
coefs, ci_lo, ci_hi, pvals, labels = [], [], [], [], []
for lab, col in measures.items():
    tmp = val[["instability", col]].copy()
    tmp["inst_lag1"] = tmp["instability"].shift(1)
    tmp["x_lag1"] = tmp[col].shift(1)
    tmp = tmp.dropna()
    Xv = pd.DataFrame({"const": 1.0,
                       "predictor": tmp["x_lag1"].astype(float),
                       "inst_lag1": tmp["inst_lag1"].astype(float)})
    res = sm.OLS(tmp["instability"].astype(float), Xv).fit()
    coefs.append(res.params["predictor"])
    ci = res.conf_int().loc["predictor"]
    ci_lo.append(ci[0]); ci_hi.append(ci[1])
    pvals.append(res.pvalues["predictor"]); labels.append(lab)
    print(f"{lab.replace(chr(10), ' ')}: beta={coefs[-1]:.3f} "
          f"p={pvals[-1]:.4f} R²={res.rsquared:.3f}")

import matplotlib.gridspec as gridspec
fig = plt.figure(figsize=(16, 6))
gs = gridspec.GridSpec(1, 2, width_ratios=[1.4, 1])
ax1 = fig.add_subplot(gs[0])
ax1.plot(val["month"], znorm(val["nv"]), color="steelblue",
         linewidth=2, label="Static Entropy", zorder=5)
ax1.plot(val["month"], znorm(val["nv_roll_sd_3"]), "--", color="coral",
         linewidth=1.8, alpha=0.85, label="Rolling SD (3-month)")
ax1.plot(val["month"], znorm(val["instability"]), ":", color="grey",
         linewidth=1.2, alpha=0.7, label="Political Instability")
ax1.axhline(0, color="black", linewidth=0.5)
ax1.set_xlabel("Month"); ax1.set_ylabel("Standardized value (z-score)")
ax1.set_title("Static Entropy vs Rolling SD Over Time", fontweight="bold")
ax1.legend(fontsize=9, loc="upper right"); ax1.grid(True, alpha=0.3)

ax2 = fig.add_subplot(gs[1])
x = np.arange(len(labels))
barcolors = ["steelblue" if pp < 0.05 else "lightgrey" for pp in pvals]
bars = ax2.bar(x, coefs, color=barcolors, alpha=0.85, width=0.55,
               edgecolor="black", linewidth=0.5)
ax2.errorbar(x, coefs,
             yerr=[[c - l for c, l in zip(coefs, ci_lo)],
                   [u - c for c, u in zip(coefs, ci_hi)]],
             fmt="none", color="black", capsize=4, linewidth=1.2)
ax2.axhline(0, color="black", linewidth=1)
ax2.set_xticks(x); ax2.set_xticklabels(labels, fontsize=9)
ax2.set_ylabel(r"$\beta$ coefficient (predicting instability)")
ax2.set_title("Predictive Validity Against\nPolitical Instability",
              fontweight="bold")
for bar, pp in zip(bars, pvals):
    sig = ("***" if pp < 0.01 else "**" if pp < 0.05
           else "*" if pp < 0.10 else "n.s.")
    ax2.text(bar.get_x() + bar.get_width() / 2, max(coefs) + 0.05, sig,
             ha="center", va="bottom", fontsize=10,
             color="red" if pp < 0.05 else "grey")
ax2.grid(True, alpha=0.3, axis="y")
plt.tight_layout(); savefig("fig18_construct_validation"); plt.show()

In [ ]:
# Instability-by-polarization interaction in the dynamic regression
d = df.copy()
d["nv_lag1"] = d["nv"].shift(1)
d["inst_l1"] = znorm(d["instability"]).shift(1)
d["pol_l1"] = znorm(d["geo_ag_polarization"]).shift(1)
d["inter_l1"] = d["inst_l1"] * d["pol_l1"]
d = d.dropna(subset=["nv", "nv_lag1", "inst_l1", "pol_l1", "inter_l1"])
X = sm.add_constant(d[["nv_lag1", "inst_l1", "pol_l1", "inter_l1"]].astype(float))
for label, kw in [("OLS", {}), ("HAC", dict(cov_type="HAC",
                                            cov_kwds={"maxlags": 3}))]:
    res = sm.OLS(d["nv"], X).fit(**kw)
    print(f"--- interaction model ({label}) ---")
    print(res.summary().tables[1])

In [ ]:
# Additional specifications reported in the paper (Table 2)
print("=" * 60); print("A) Pandemic and seasonality controls")
dA = df.copy()
dA["covid"] = ((dA["month"] >= "2020-03-01") & (dA["month"] <= "2021-06-01")).astype(int)
dA["sin12"] = np.sin(2 * np.pi * dA["month"].dt.month / 12)
dA["cos12"] = np.cos(2 * np.pi * dA["month"].dt.month / 12)
dA["inst_l1"] = dA["instability"].shift(1)
dA = dA.dropna(subset=["nv", "nv_lag1", "inst_l1"])
XA = sm.add_constant(dA[["nv_lag1", "inst_l1", "covid", "sin12", "cos12"]].astype(float))
print(sm.OLS(dA["nv"], XA).fit(cov_type="HAC",
      cov_kwds={"maxlags": 3}).summary().tables[1])

print("=" * 60); print("B) Flexible volume controls")
dA["vol_z"] = znorm(dA["total_events"])
dA["vol_z2"] = dA["vol_z"] ** 2
XB = sm.add_constant(dA[["nv_lag1", "inst_l1", "vol_z", "vol_z2"]].astype(float))
print(sm.OLS(dA["nv"], XB).fit(cov_type="HAC",
      cov_kwds={"maxlags": 3}).summary().tables[1])


In [ ]:
# Monitoring figure (Discussion): fragmentation series, regime timeline,
# Executive suspensions, and documented episodes. Requires the DataFrame
# `rdf` with columns `month`, `nv`, `regime` from the regime cells above.


import matplotlib.patches as mpatches
import matplotlib.pyplot as plt
import pandas as pd


# Check that the regime cells have been run.
if "rdf" not in globals():
    raise RuntimeError(
        "`rdf` not found. Run the regime cells above first."
    )

required_columns = {"month", "nv", "regime"}
missing_columns = required_columns.difference(rdf.columns)
if missing_columns:
    raise KeyError(
        "Missing columns in rdf: " + ", ".join(sorted(missing_columns))
    )

# In the notebook, regime is coded 0/1/2; the figure uses the paper's
# convention: 1 = baseline, 2 = acute co-occurrence, 3 = elevated polarization.
df_monitoring = rdf[["month", "nv", "regime"]].copy()
df_monitoring["month"] = pd.to_datetime(df_monitoring["month"])
df_monitoring["regime"] = df_monitoring["regime"].astype(int) + 1
df_monitoring = df_monitoring.set_index("month").sort_index()

COL_NF = "nv"
COL_REGIME = "regime"

REGIME_COLORS = {2: "#f3c1bb", 3: "#f7e3b5"}
REGIME_NAMES = {
    2: "Acute co-occurrence regime",
    3: "Elevated polarization regime",
}

SUSPENSIONS = [
    ("2017-01-01", "2019-12-31"),
    ("2022-02-01", "2024-01-31"),
]

EPISODES = [
    ("2016-06-01", "Brexit\nreferendum"),
    ("2017-01-01", "RHI crisis /\ndFM resignation"),
    ("2018-09-01", "Salzburg /\nChequers"),
    ("2019-03-01", "Meaningful\nvotes"),
    ("2020-09-01", "Internal\nMarket Bill"),
    ("2022-02-01", "Executive\ncollapse"),
    ("2022-07-01", "NI Protocol\nBill"),
    ("2024-02-01", "Executive\nrestored"),
]

fig = plt.figure(figsize=(16, 7.5))
grid = fig.add_gridspec(2, 1, height_ratios=[12, 0.8], hspace=0.06)
ax = fig.add_subplot(grid[0])
ax_suspension = fig.add_subplot(grid[1], sharex=ax)

# Bandes de régime : uniquement les périodes aiguës et polarisées.
reg = df_monitoring[COL_REGIME]
run_start = df_monitoring.index[0]
for i in range(1, len(df_monitoring) + 1):
    end_of_run = i == len(df_monitoring) or reg.iloc[i] != reg.iloc[i - 1]
    if end_of_run:
        regime_value = int(reg.iloc[i - 1])
        if regime_value in REGIME_COLORS:
            right = (
                df_monitoring.index[i]
                if i < len(df_monitoring)
                else df_monitoring.index[-1] + pd.offsets.MonthBegin(1)
            )
            ax.axvspan(
                run_start,
                right,
                color=REGIME_COLORS[regime_value],
                lw=0,
                alpha=0.72,
                zorder=0,
            )
        if i < len(df_monitoring):
            run_start = df_monitoring.index[i]

# Panneau séparé : suspensions de l'Exécutif.
ax_suspension.set_ylim(0, 1)
for start, end in SUSPENSIONS:
    start = pd.Timestamp(start)
    end = pd.Timestamp(end)
    ax_suspension.axvspan(
        start,
        end,
        color="#5a5a5a",
        lw=0,
        zorder=1,
    )
    ax_suspension.text(
        start + (end - start) / 2,
        0.5,
        "Executive suspension",
        ha="center",
        va="center",
        fontsize=8.5,
        color="white",
        fontweight="bold",
        zorder=2,
    )

# Fragmentation mensuelle et moyenne mobile centrée sur trois mois.
ax.plot(
    df_monitoring.index,
    df_monitoring[COL_NF],
    color="#9bb0c9",
    lw=0.9,
    zorder=2,
)
ax.plot(
    df_monitoring.index,
    df_monitoring[COL_NF].rolling(3, center=True).mean(),
    color="#1f4e79",
    lw=2.2,
    zorder=3,
)

# Épisodes documentés, avec étiquettes placées au-dessus des données.
y0 = df_monitoring[COL_NF].min()
y1 = df_monitoring[COL_NF].max()
data_range = y1 - y0
ax.set_ylim(y0 - 0.06 * data_range, y1 + 0.24 * data_range)

for position, (date, label) in enumerate(EPISODES):
    date = pd.Timestamp(date)
    ax.axvline(
        date, color="#555555", lw=0.8, ls=(0, (2, 3)), alpha=0.75, zorder=4
    )
    label_y = y1 + (0.17 if position % 2 == 0 else 0.08) * data_range
    ax.annotate(
        label,
        xy=(date, label_y),
        xytext=(date, label_y),
        ha="center",
        va="center",
        fontsize=8,
        color="#222222",
        zorder=5,
        bbox=dict(
            boxstyle="round,pad=0.22",
            facecolor="white",
            edgecolor="#b0b0b0",
            linewidth=0.6,
            alpha=0.92,
        ),
    )

handles = [
    plt.Line2D(
        [], [], color="#1f4e79", lw=2.2,
        label="Narrative fragmentation (3-month rolling mean)",
    ),
    plt.Line2D(
        [], [], color="#9bb0c9", lw=0.9,
        label="Narrative fragmentation (monthly)",
    ),
    mpatches.Patch(color=REGIME_COLORS[2], label=REGIME_NAMES[2]),
    mpatches.Patch(color=REGIME_COLORS[3], label=REGIME_NAMES[3]),
    mpatches.Patch(color="#5a5a5a", label="Executive suspension (bottom band)"),
]

fig.legend(
    handles=handles,
    loc="lower center",
    bbox_to_anchor=(0.5, 0.01),
    fontsize=9,
    frameon=False,
    ncol=3,
)
ax.set_title(
    "Narrative Fragmentation, Political–Discursive Regimes, and Key Episodes",
    fontsize=14,
    fontweight="bold",
    pad=15,
)
ax.set_ylabel("Narrative fragmentation (Shannon entropy)")
ax.set_xlim(
    df_monitoring.index[0],
    df_monitoring.index[-1] + pd.offsets.MonthBegin(1),
)
ax.margins(x=0)
ax.grid(axis="y", color="#d9d9d9", linewidth=0.7, alpha=0.65)
for spine in ("top", "right"):
    ax.spines[spine].set_visible(False)

# Habillage du panneau des suspensions.
ax_suspension.set_yticks([])
ax_suspension.set_ylabel("Executive\nstatus", rotation=0, ha="right", va="center",
                         fontsize=8.5, labelpad=12)
ax_suspension.tick_params(axis="x", labelsize=9)
for spine in ("top", "right", "left"):
    ax_suspension.spines[spine].set_visible(False)
plt.setp(ax.get_xticklabels(), visible=False)

fig.subplots_adjust(left=0.08, right=0.985, top=0.88, bottom=0.18)
plt.savefig(FIGURES_DIR / "monitoring_dashboard.png", dpi=300, bbox_inches="tight")
plt.show()